# 이진분류 v10 — 인프라 전달용 독립 추론 노트북

무패턴 포함 학습 모델입니다. 전처리부터 JSON 응답까지 필요한 코드를 이 노트북에 직접 포함했습니다. 다른 실험 소스나 `.py` 모듈은 필요하지 않습니다. 이 폴더 전체를 복사하고 위에서 아래로 실행하세요. 학습 가중치와 피처 순서는 변경하지 않았습니다.

Python 3.12 환경에서 `python -m pip install -r requirements.txt`로 설치합니다. CPU/CUDA PyTorch 배포판은 인프라 환경에 맞추세요. 기본 실행은 CPU입니다. 이 노트북은 추론용이며 모델 재학습과 XAI는 포함하지 않습니다.

## 1. 설정과 입력 계약

`settings.json`에서 입력 Parquet, 기준일, 예측 대상 기간을 지정합니다. 거래 ID·계좌 ID는 정수입니다. transactions는 edge_id, ts, src, dst, paid, received, payment_currency, receiving_currency, payment_format, duplicate_group_id가 필요합니다. KYC는 node_id, entity_id, entity_type, country가 필요합니다. 학습 정답은 입력하지 않습니다.

기준일까지의 누적 거래를 공급해야 하며 단일 거래만으로 과거 이력을 대체할 수 없습니다. 원천 시각의 시간대를 확인해 일관되게 변환하고, 시간대 정보가 없는 HI-Large 원본을 임의로 UTC라고 간주하지 마세요. 스냅샷은 기준일 전체를 포함하므로 거래 시점 실시간 인과 추론이 아닙니다.

In [ ]:
from pathlib import Path
import json

# 노트북 폴더에서 실행하는 것이 기본입니다. 상위 전달 폴더에서도 실행 가능합니다.
ROOT = Path.cwd().resolve()
if not (ROOT / "models" / "binary.pt").is_file():
    ROOT = ROOT / "01_binary_v10"
if not (ROOT / "models" / "binary.pt").is_file():
    raise FileNotFoundError("노트북이 있는 모델 폴더를 작업 디렉터리로 지정하세요.")
FAMILY = "binary"
import os
DEVICE = os.environ.get("AML_DEVICE", json.loads((ROOT / "settings.json").read_text()).get("device", "cpu"))


In [ ]:
"""Feature order is part of the model contract; never reorder columns."""

import json
from pathlib import Path

SCHEMA = json.loads((ROOT / "assets" / "feature_schema.json").read_text())
NODE_FEATURES = SCHEMA["node_features"]
EDGE_FEATURES = SCHEMA["edge_features"]
ENTITY_TYPES = SCHEMA["entity_types"]
API_CLASSES = SCHEMA["pattern_classes"]


In [ ]:
"""Validated relative paths and atomic JSON artifacts."""

import hashlib
import json
from pathlib import Path


In [ ]:
def resolve_path(value):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (ROOT / path).resolve()


In [ ]:
def load_settings(path=None):
    settings = json.loads((Path(path) if path else ROOT / "settings.json").read_text())
    for key in ["transactions", "kyc", "output_dir"]:
        settings[key] = str(resolve_path(settings[key]))
    return settings


In [ ]:
def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False))
    temp.replace(path)


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


In [ ]:
def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


In [ ]:
q = sql_string
settings = load_settings()
settings["device"] = DEVICE
print({k: settings[k] for k in ["context_end", "target_start", "target_end", "device"]})


## 2. 고정 피처와 전처리

노드 19개·거래 17개 피처 및 보정 통계를 그대로 사용합니다. 금액 통계를 새 입력에 맞춰 재학습하지 않습니다. 정확 중복은 입력 duplicate_group_id에 지정된 대표 거래로 처리합니다. 시각 기반 이력은 날짜·시간으로 계산하고, 거래 연결은 행 위치가 아닌 edge_id를 사용합니다. 함수 정의 셀은 아래 예제 실행 전까지 데이터를 처리하지 않습니다.

In [ ]:
"""Original cumulative node feature formulas, selected into frozen 19-column order."""

import numpy as np
import pandas as pd

ALL_FEATURES = (
    [
        "log_out_count",
        "log_in_count",
        "log_out_neighbors",
        "log_in_neighbors",
        "log_self_count",
        "log_reciprocal_neighbors",
        "net_count_ratio",
        "reciprocity_ratio",
        "mean_paid_currency_z",
        "mean_received_currency_z",
        "paid_outlier_fraction",
        "received_outlier_fraction",
        "log_currency_count",
        "log_active_days",
    ]
    + ["type_" + x for x in ENTITY_TYPES]
    + [
        "log_entity_observed_accounts",
        "log_entity_out_diversity",
        "log_entity_in_diversity",
    ]
)


In [ ]:
def finite(name, x):
    if not np.isfinite(x).all():
        raise ValueError(name + " contains nonfinite values")
    return x


In [ ]:
def build_nodes(c):
    agg = c.execute(
        "WITH outgoing AS (SELECT src AS node_id,count(*) AS out_n,count(DISTINCT dst) AS out_u,\n          count(*) FILTER(WHERE src=dst) AS self_n,avg(paid_currency_z) AS out_z,\n          avg(CASE WHEN abs(paid_currency_z)>=3 THEN 1.0 ELSE 0 END) AS out_tail FROM hist GROUP BY src),\n      incoming AS (SELECT dst AS node_id,count(*) AS in_n,count(DISTINCT src) AS in_u,avg(received_currency_z) AS in_z,\n          avg(CASE WHEN abs(received_currency_z)>=3 THEN 1.0 ELSE 0 END) AS in_tail FROM hist GROUP BY dst),\n      pairs AS (SELECT DISTINCT src,dst FROM hist WHERE src!=dst),\n      reciprocal AS (SELECT a.src AS node_id,count(*) AS reciprocal_n FROM pairs a JOIN pairs b ON a.src=b.dst AND a.dst=b.src GROUP BY a.src),\n      currency_set AS (SELECT src AS node_id,payment_currency AS currency FROM hist UNION SELECT dst,receiving_currency FROM hist),\n      currencies AS (SELECT node_id,count(*) AS n_currency FROM currency_set GROUP BY node_id),\n      active AS (SELECT src AS node_id,day FROM hist UNION SELECT dst,day FROM hist),\n      active_days AS (SELECT node_id,count(*) AS n_days FROM active GROUP BY node_id),\n      ent_pairs AS (SELECT DISTINCT source_entity,target_entity FROM hist WHERE source_entity IS NOT NULL AND target_entity IS NOT NULL),\n      ent_out AS (SELECT source_entity AS entity_id,count(*) AS ent_out_n FROM ent_pairs GROUP BY source_entity),\n      ent_in AS (SELECT target_entity AS entity_id,count(*) AS ent_in_n FROM ent_pairs GROUP BY target_entity)\n      SELECT k.*,CASE WHEN k.entity_id IS NULL THEN 0 ELSE count(*) OVER(PARTITION BY k.entity_id) END AS observed_entity_accounts,coalesce(o.out_n,0) AS out_n,coalesce(i.in_n,0) AS in_n,coalesce(o.out_u,0) AS out_u,\n       coalesce(i.in_u,0) AS in_u,coalesce(o.self_n,0) AS self_n,coalesce(r.reciprocal_n,0) AS reciprocal_n,\n       coalesce(o.out_z,0) AS out_z,coalesce(i.in_z,0) AS in_z,coalesce(o.out_tail,0) AS out_tail,\n       coalesce(i.in_tail,0) AS in_tail,cu.n_currency,ad.n_days,coalesce(eo.ent_out_n,0) AS ent_out_n,\n       coalesce(ei.ent_in_n,0) AS ent_in_n\n      FROM active_days ad JOIN kyc k USING(node_id) LEFT JOIN outgoing o USING(node_id)\n      LEFT JOIN incoming i USING(node_id) LEFT JOIN reciprocal r USING(node_id)\n      JOIN currencies cu USING(node_id) LEFT JOIN ent_out eo ON k.entity_id=eo.entity_id\n      LEFT JOIN ent_in ei ON k.entity_id=ei.entity_id ORDER BY node_id"
    ).fetchdf()
    if agg.empty:
        raise ValueError("No graph history")
    v = lambda name: agg[name].to_numpy(np.float64)
    outn, inn = (v("out_n"), v("in_n"))
    ou, iu = (v("out_u"), v("in_u"))
    rec = v("reciprocal_n")
    fields = [
        np.log1p(outn),
        np.log1p(inn),
        np.log1p(ou),
        np.log1p(iu),
        np.log1p(v("self_n")),
        np.log1p(rec),
        (outn - inn) / np.maximum(outn + inn, 1),
        rec / np.maximum(ou + iu, 1),
        v("out_z"),
        v("in_z"),
        v("out_tail"),
        v("in_tail"),
        np.log1p(v("n_currency")),
        np.log1p(v("n_days")),
    ]
    fields += [(agg.entity_type == typ).to_numpy(np.float64) for typ in ENTITY_TYPES]
    fields += [
        np.log1p(v("observed_entity_accounts")),
        np.log1p(v("ent_out_n")),
        np.log1p(v("ent_in_n")),
    ]
    nx = finite("daily node features", np.stack(fields, axis=1).astype(np.float32))
    node_ids = agg.node_id.to_numpy(np.int64)
    out = pd.DataFrame(
        nx[:, [ALL_FEATURES.index(n) for n in NODE_FEATURES]], columns=NODE_FEATURES
    )
    out.insert(0, "node_id", node_ids)
    return out


In [ ]:
"""Label-free exact-deduplicated node19/edge17 preprocessing with frozen calibration.

Inputs are normalized transaction/KYC Parquet, not a vendor-specific raw CSV.
All historical transactions through context_end must be supplied. Snapshot
features include that whole day: daily close semantics, never streaming claims.
"""

from pathlib import Path
import json
import duckdb
import pandas as pd

TX_COLUMNS = [
    "edge_id",
    "ts",
    "src",
    "dst",
    "paid",
    "received",
    "payment_currency",
    "receiving_currency",
    "payment_format",
    "duplicate_group_id",
]
KYC_COLUMNS = ["node_id", "entity_id", "entity_type", "country"]


In [ ]:
def amount(column, prefix):
    return f"CASE WHEN {prefix}.scale IS NULL THEN 0 ELSE greatest(-8,least(8,(ln(1+t.{column}::DOUBLE/{prefix}.scale)-{prefix}.med)/{prefix}.iqr)) END"


In [ ]:
def validate_inputs(c, transactions, kyc):
    """Reject broken identifiers and missing KYC rather than silently losing rows."""
    c.execute(f"CREATE VIEW raw_tx AS SELECT * FROM read_parquet({q(transactions)})")
    c.execute(f"CREATE VIEW raw_kyc AS SELECT * FROM read_parquet({q(kyc)})")
    for table, columns in [("raw_tx", TX_COLUMNS), ("raw_kyc", KYC_COLUMNS)]:
        actual = set(c.execute(f"DESCRIBE {table}").fetchdf().column_name)
        if missing := (set(columns) - actual):
            raise ValueError(f"{table}: missing {sorted(missing)}")
    types = dict((row[:2] for row in c.execute("DESCRIBE raw_tx").fetchall()))
    for name in ["edge_id", "src", "dst", "duplicate_group_id"]:
        if types[name] not in ["BIGINT", "INTEGER", "SMALLINT", "TINYINT"]:
            raise ValueError(f"{name} must be a signed integer column")
    if not types["ts"].startswith("TIMESTAMP") or "WITH TIME ZONE" in types["ts"]:
        raise ValueError("ts must be a UTC-normalized timezone-naive Parquet timestamp")
    kyc_types = dict((row[:2] for row in c.execute("DESCRIBE raw_kyc").fetchall()))
    if kyc_types["node_id"] not in ["BIGINT", "INTEGER", "SMALLINT", "TINYINT"]:
        raise ValueError("KYC node_id must be a signed integer column")
    if c.execute("SELECT count(*)-count(DISTINCT edge_id) FROM raw_tx").fetchone()[0]:
        raise ValueError("edge_id must be unique")
    if c.execute("SELECT count(*)-count(DISTINCT node_id) FROM raw_kyc").fetchone()[0]:
        raise ValueError("node_id must be unique")
    bad = c.execute(
        "SELECT count(*) FROM raw_tx WHERE edge_id IS NULL OR src IS NULL OR dst IS NULL OR ts IS NULL OR edge_id<0 OR src<0 OR dst<0 OR paid IS NULL OR received IS NULL OR NOT isfinite(paid::DOUBLE) OR NOT isfinite(received::DOUBLE) OR paid<0 OR received<0 OR payment_currency IS NULL OR receiving_currency IS NULL"
    ).fetchone()[0]
    if bad:
        raise ValueError(f"{bad} transactions have invalid required fields")
    if c.execute(
        "SELECT count(*) FROM raw_kyc WHERE node_id IS NULL OR node_id<0"
    ).fetchone()[0]:
        raise ValueError("Invalid KYC node_id")
    missing = c.execute(
        "SELECT count(*) FROM (SELECT src AS node_id FROM raw_tx UNION SELECT dst FROM raw_tx) n ANTI JOIN raw_kyc k USING(node_id)"
    ).fetchone()[0]
    if missing:
        raise ValueError(f"{missing} accounts missing from KYC mapping")
    if c.execute(
        "SELECT count(*) FROM raw_tx a ANTI JOIN raw_tx b ON a.duplicate_group_id=b.edge_id WHERE a.duplicate_group_id IS NOT NULL"
    ).fetchone()[0]:
        raise ValueError("Duplicate representative missing from input history")
    fields = [
        "ts",
        "src",
        "dst",
        "paid",
        "received",
        "payment_currency",
        "receiving_currency",
        "payment_format",
    ]
    different = " OR ".join((f"a.{name} IS DISTINCT FROM b.{name}" for name in fields))
    invalid = c.execute(
        f"SELECT count(*) FROM raw_tx a JOIN raw_tx b ON a.duplicate_group_id=b.edge_id WHERE ({different}) OR (b.duplicate_group_id IS NOT NULL AND b.duplicate_group_id!=b.edge_id)"
    ).fetchone()[0]
    if invalid:
        raise ValueError(
            "Duplicate group must reference an identical retained transaction"
        )


In [ ]:
def prepare_snapshot(settings):
    """Write feature/graph Parquet and a completion manifest; no labels are read."""
    root = Path(settings["output_dir"]) / "prepared"
    if root.exists():
        raise FileExistsError(
            f"{root}: use a fresh output directory, or explicitly remove generated outputs"
        )
    end = pd.Timestamp(settings["context_end"])
    start = pd.Timestamp(settings["target_start"])
    target_end = pd.Timestamp(settings["target_end"])
    if not start <= target_end <= end:
        raise ValueError("Require target_start <= target_end <= context_end")
    if settings.get("neighbors", 100) != 100 or settings.get("sampling_seed", 42) != 42:
        raise ValueError("Frozen models require neighbors=100 and sampling_seed=42")
    root.mkdir(parents=True)
    c = duckdb.connect(str(root / "work.duckdb"))
    try:
        c.execute(
            f"SET threads={int(settings.get('threads', 2))}; SET memory_limit={q(settings.get('memory_limit', '2GB'))}"
        )
        validate_inputs(c, settings["transactions"], settings["kyc"])
        cols = ",".join(TX_COLUMNS)
        c.execute(
            f"CREATE TABLE tx AS SELECT {cols},ts::DATE AS day FROM raw_tx WHERE (duplicate_group_id IS NULL OR edge_id=duplicate_group_id) AND ts::DATE<={q(str(end.date()))} ORDER BY edge_id"
        )
        if not c.execute("SELECT count(*) FROM tx").fetchone()[0]:
            raise ValueError("Empty snapshot")
        c.execute(
            "CREATE TABLE kyc AS SELECT node_id,entity_id,coalesce(entity_type,'UNKNOWN') AS entity_type,coalesce(country,'UNKNOWN') AS country FROM raw_kyc"
        )
        c.execute(
            f"CREATE VIEW stats AS SELECT * FROM read_parquet({q(ROOT / 'assets/currency_stats.parquet')})"
        )
        c.execute(
            f"CREATE VIEW amount_stats AS SELECT * FROM read_parquet({q(ROOT / 'assets/amount_stats.parquet')})"
        )
        c.execute(
            "CREATE TABLE pair_state AS SELECT src,dst,day,\n          sum(n) OVER(PARTITION BY src,dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS history_count,\n          max(last_ts) OVER(PARTITION BY src,dst ORDER BY day ROWS UNBOUNDED PRECEDING) AS last_ts\n          FROM (SELECT src,dst,day,count(*) n,max(ts) last_ts FROM tx GROUP BY src,dst,day)"
        )
        c.execute(
            "CREATE TABLE pairs AS SELECT t.src,t.dst,t.day,coalesce(f.history_count,0) pair_n,coalesce(r.history_count,0) reverse_n,\n          CASE WHEN f.last_ts IS NULL THEN -1 ELSE date_diff('minute',f.last_ts,t.day::TIMESTAMP) END age\n          FROM (SELECT DISTINCT src,dst,day FROM tx) t\n          ASOF LEFT JOIN pair_state f ON t.src=f.src AND t.dst=f.dst AND t.day>f.day\n          ASOF LEFT JOIN pair_state r ON t.src=r.dst AND t.dst=r.src AND t.day>r.day"
        )
        windows = []
        for name, part in [("pair", "src,dst"), ("sender", "src"), ("receiver", "dst")]:
            windows.extend(
                [
                    f"count(*) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) {name}_prior",
                    f"max(ts) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN UNBOUNDED PRECEDING AND INTERVAL '1 minute' PRECEDING) {name}_last",
                    f"count(*) OVER(PARTITION BY {part} ORDER BY ts RANGE BETWEEN INTERVAL '60 minutes' PRECEDING AND INTERVAL '1 minute' PRECEDING) {name}_hour",
                ]
            )
        c.execute(
            "CREATE TABLE temporal AS SELECT edge_id," + ",".join(windows) + " FROM tx"
        )
        gap = (
            lambda n: f"CASE WHEN z.{n}_last IS NULL THEN 0 ELSE ln(2+date_diff('minute',z.{n}_last,t.ts)) END"
        )
        expressions = [
            "ln(1+z.pair_prior)",
            gap("pair"),
            "ln(1+z.pair_hour)",
            gap("sender"),
            "ln(1+z.sender_hour)",
            gap("receiver"),
            "ln(1+z.receiver_hour)",
            amount("paid", "ps"),
            "ln(1+p.pair_n)",
            "ln(1+p.reverse_n)",
            "CASE WHEN p.age<0 THEN 0 ELSE ln(2+p.age) END",
            "CASE WHEN sk.entity_id IS NOT NULL AND sk.entity_id=dk.entity_id THEN 1 ELSE 0 END",
            "CASE WHEN sk.country!='UNKNOWN' AND dk.country!='UNKNOWN' AND sk.country=dk.country THEN 1 ELSE 0 END",
            "CASE WHEN sk.country!='UNKNOWN' AND dk.country!='UNKNOWN' THEN 1 ELSE 0 END",
            "CASE WHEN t.payment_currency=t.receiving_currency THEN 1 ELSE 0 END",
            "CASE WHEN t.src=t.dst THEN 1 ELSE 0 END",
            "CASE WHEN a.m IS NULL OR a.sd IS NULL THEN 0 ELSE greatest(-10,least(10,(ln(1+t.paid::DOUBLE)-a.m)/(a.sd+0.000001))) END",
        ]
        features = ",".join(
            (
                f'({value})::FLOAT AS "{name}"'
                for name, value in zip(EDGE_FEATURES, expressions)
            )
        )
        c.execute(
            f"CREATE TABLE edge_features AS SELECT t.edge_id,{features} FROM tx t JOIN temporal z USING(edge_id)\n          JOIN pairs p ON t.src=p.src AND t.dst=p.dst AND t.day=p.day JOIN kyc sk ON sk.node_id=t.src JOIN kyc dk ON dk.node_id=t.dst\n          LEFT JOIN stats ps ON ps.currency=t.payment_currency LEFT JOIN amount_stats a ON a.payment_currency=t.payment_currency ORDER BY t.edge_id"
        )
        c.execute(
            f"CREATE VIEW hist AS SELECT t.*,{amount('paid', 'ps')} paid_currency_z,{amount('received', 'rs')} received_currency_z,\n          ks.entity_id source_entity,kd.entity_id target_entity FROM tx t JOIN kyc ks ON ks.node_id=t.src JOIN kyc kd ON kd.node_id=t.dst\n          LEFT JOIN stats ps ON ps.currency=t.payment_currency LEFT JOIN stats rs ON rs.currency=t.receiving_currency"
        )
        nodes = build_nodes(c)
        c.register("node_features", nodes)
        adjacency = []
        for direction, owner in [("incoming", "dst"), ("outgoing", "src")]:
            adjacency.append(
                f"SELECT edge_id,{owner} AS owner,{q(direction)} AS direction FROM tx QUALIFY row_number() OVER(PARTITION BY {owner} ORDER BY hash(edge_id,{owner},42,DATE {q(str(end.date()))}),edge_id)<=100"
            )
        c.execute("CREATE TABLE adjacency AS " + " UNION ALL ".join(adjacency))
        c.execute(
            f"CREATE TABLE targets AS SELECT edge_id FROM tx WHERE day BETWEEN {q(str(start.date()))} AND {q(str(target_end.date()))} ORDER BY edge_id"
        )
        if not c.execute("SELECT count(*) FROM targets").fetchone()[0]:
            raise ValueError("No target transactions in selected dates")
        bad = " OR ".join(
            (f'"{n}" IS NULL OR NOT isfinite("{n}")' for n in EDGE_FEATURES)
        )
        if c.execute("SELECT count(*) FROM edge_features WHERE " + bad).fetchone()[0]:
            raise ValueError("Nonfinite edge features")
        orders = {
            "tx": "edge_id",
            "edge_features": "edge_id",
            "node_features": "node_id",
            "adjacency": "direction,owner,edge_id",
            "targets": "edge_id",
        }
        counts = {}
        for name, order in orders.items():
            counts[name] = c.execute(f"SELECT count(*) FROM {name}").fetchone()[0]
            c.execute(
                f"COPY (SELECT * FROM {name} ORDER BY {order}) TO {q(root / (name + '.parquet'))} (FORMAT PARQUET, COMPRESSION ZSTD)"
            )
        manifest = dict(
            schema_version="node19_edge17_v1",
            counts=counts,
            context_end=str(end.date()),
            target_start=str(start.date()),
            target_end=str(target_end.date()),
            history_start=str(c.execute("SELECT min(day) FROM tx").fetchone()[0]),
            neighbors=100,
            sampling_seed=42,
            duckdb_version=duckdb.__version__,
            labels_used=False,
            context_policy="cumulative through context_end inclusive; not transaction-time causal",
            inputs={
                k: dict(path=settings[k], sha256=sha256(settings[k]))
                for k in ["transactions", "kyc"]
            },
            calibration={
                p.name: sha256(p)
                for p in [
                    ROOT / "assets/currency_stats.parquet",
                    ROOT / "assets/amount_stats.parquet",
                ]
            },
        )
        write_json(root / "manifest.json", manifest)
    finally:
        c.close()
    (root / "work.duckdb").unlink(missing_ok=True)
    return root


In [ ]:
def ensure_snapshot(settings):
    """Reuse only a completed snapshot with identical inputs, calibration and settings."""
    root = Path(settings["output_dir"]) / "prepared"
    if not root.exists():
        return prepare_snapshot(settings)
    manifest_path = root / "manifest.json"
    if not manifest_path.exists():
        raise ValueError("Incomplete output directory: select a fresh output_dir")
    manifest = json.loads(manifest_path.read_text())
    for key in [
        "context_end",
        "target_start",
        "target_end",
        "neighbors",
        "sampling_seed",
    ]:
        if manifest[key] != settings[key]:
            raise ValueError(
                f"Existing snapshot differs in {key}: select a fresh output_dir"
            )
    for key in ["transactions", "kyc"]:
        if manifest["inputs"][key]["sha256"] != sha256(settings[key]):
            raise ValueError(f"Changed input {key}: select a fresh output_dir")
    for name, expected in manifest["calibration"].items():
        if sha256(ROOT / "assets" / name) != expected:
            raise ValueError("Calibration changed: select a fresh output_dir")
    for name in ["tx", "edge_features", "node_features", "adjacency", "targets"]:
        if not (root / f"{name}.parquet").is_file():
            raise ValueError(f"Missing prepared artifact: {name}")
    return root


## 3. 거래별 2홉 그래프

입금·출금 방향 각각 최대 100개, sampling_seed=42를 고정합니다. 양끝 계좌를 기준으로 2홉을 확장하고 ego 표시를 붙여 노드 입력은 20차원이 됩니다. 전체 누적 그래프에 존재하는 순환도 샘플링된 입력에서는 잘릴 수 있습니다.

In [ ]:
"""On-demand, two-hop disjoint transaction graph; avoids loading full history into RAM."""

from dataclasses import dataclass
from pathlib import Path
import json
import duckdb, numpy as np, pandas as pd, torch


In [ ]:
@dataclass
class GraphInput:
    arrays: tuple
    topology: tuple
    edge_id: int
    context_end: str

    def tensors(self, device="cpu"):
        return tuple(
            (
                torch.as_tensor(
                    np.array(a, copy=True),
                    device=device,
                    dtype=(
                        torch.long
                        if i in [1, 3]
                        else torch.bool if i in [5, 6] else torch.float32
                    ),
                )
                for i, a in enumerate(self.arrays)
            )
        )

    def save(self, path):
        np.savez_compressed(
            path,
            **{f"b{i}": v for i, v in enumerate(self.arrays)},
            **{f"t{i}": v for i, v in enumerate(self.topology)},
        )


In [ ]:
def reference_graph(path, metadata):
    with np.load(path, allow_pickle=False) as data:
        return GraphInput(
            tuple((data[f"b{i}"].copy() for i in range(7))),
            tuple((data[f"t{i}"].copy() for i in range(8))),
            int(metadata["edge_id"]),
            metadata["context_end"],
        )


In [ ]:
class GraphSnapshot:

    def __init__(self, prepared):
        self.root = Path(prepared)
        self.manifest = json.loads((self.root / "manifest.json").read_text())
        if self.manifest["schema_version"] != "node19_edge17_v1":
            raise ValueError("Unsupported feature schema")
        self.c = duckdb.connect()
        self.c.execute("SET threads=2; SET memory_limit='2GB'")
        for table in ["tx", "edge_features", "node_features", "adjacency", "targets"]:
            self.c.execute(
                f"CREATE VIEW {table} AS SELECT * FROM read_parquet({q(self.root / (table + '.parquet'))})"
            )

    def close(self):
        self.c.close()

    def __enter__(self):
        return self

    def __exit__(self, *args):
        self.close()

    def target_ids(self, limit=100):
        if limit < 1:
            raise ValueError("limit must be positive")
        return (
            self.c.execute(
                f"SELECT edge_id FROM targets ORDER BY edge_id LIMIT {int(limit)}"
            )
            .fetchdf()
            .edge_id.to_numpy()
        )

    def _select(self, nodes):
        self.c.register("frontier", pd.DataFrame({"owner": np.unique(nodes)}))
        return self.c.execute(
            "SELECT a.edge_id,t.src,t.dst,bool_or(a.direction='incoming') AS incoming,bool_or(a.direction='outgoing') AS outgoing FROM adjacency a JOIN frontier f USING(owner) JOIN tx t USING(edge_id) GROUP BY a.edge_id,t.src,t.dst ORDER BY a.edge_id"
        ).fetchdf()

    def build(self, edge_id):
        target = self.c.execute(
            "SELECT t.edge_id,t.src,t.dst FROM tx t JOIN targets a USING(edge_id) WHERE t.edge_id=?",
            [int(edge_id)],
        ).fetchdf()
        if len(target) != 1:
            raise ValueError("Transaction not in declared target interval")
        src, dst = (int(target.src.iloc[0]), int(target.dst.iloc[0]))
        roots = np.unique([src, dst])
        first = self._select(roots)
        neighbors = np.unique(
            np.r_[first.src[first.incoming], first.dst[first.outgoing]]
        )
        second = self._select(neighbors)
        edges = (
            pd.concat([first, second], ignore_index=True)
            .groupby(["edge_id", "src", "dst"], as_index=False)[
                ["incoming", "outgoing"]
            ]
            .max()
            .sort_values("edge_id")
        )
        nodes = np.unique(np.r_[roots, edges.src, edges.dst])
        self.c.register("needed_nodes", pd.DataFrame({"node_id": nodes}))
        features = self.c.execute(
            "SELECT n.* FROM node_features n JOIN needed_nodes u USING(node_id) ORDER BY node_id"
        ).fetchdf()
        if not np.array_equal(features.node_id, nodes):
            raise ValueError("Missing node features")
        self.c.register(
            "needed_edges",
            pd.DataFrame({"edge_id": np.unique(np.r_[edges.edge_id, edge_id])}),
        )
        attrs = (
            self.c.execute(
                "SELECT e.* FROM edge_features e JOIN needed_edges u USING(edge_id) ORDER BY edge_id"
            )
            .fetchdf()
            .set_index("edge_id")
        )
        ego = np.isin(nodes, roots).astype(np.float32)
        x = np.column_stack([features[NODE_FEATURES].to_numpy(np.float32), ego])
        ei = np.stack(
            [np.searchsorted(nodes, edges.src), np.searchsorted(nodes, edges.dst)]
        )
        ti = np.array([[np.searchsorted(nodes, src)], [np.searchsorted(nodes, dst)]])
        arrays = (
            x,
            ei,
            attrs.loc[edges.edge_id, EDGE_FEATURES].to_numpy(np.float32),
            ti,
            attrs.loc[[edge_id], EDGE_FEATURES].to_numpy(np.float32),
            edges.incoming.to_numpy(bool),
            edges.outgoing.to_numpy(bool),
        )
        flags = arrays[5].astype(np.int64) + 2 * arrays[6].astype(np.int64)
        top = (
            nodes,
            edges.edge_id.to_numpy(np.int64),
            ei[0],
            ei[1],
            ti[0],
            ti[1],
            flags,
            ego,
        )
        return GraphInput(arrays, top, int(edge_id), self.manifest["context_end"])


## 4. 모델 정의와 가중치 로딩

공통 인코더와 분류 헤드를 분리했습니다. 가중치는 체크섬 검증 후 strict=True로 로딩하고 eval 모드에서 실행합니다.

출력은 정상/자금세탁 위험 점수입니다. 기본 임계값은 검증에서 고정한 0.4338742792606354입니다.

In [ ]:
from dataclasses import dataclass
import numpy as np
import torch
from torch_geometric.nn import TransformerConv


@dataclass(frozen=True)
class Options:
    neighbors: int = 100
    outgoing: bool = True
    reverse_mp: bool = True
    ego: bool = False
    ports: bool = False
    seed: int = 42


class GraphEncoder(torch.nn.Module):
    """Checkpoint-compatible two-layer encoder with separate message directions."""

    def __init__(self, options):
        super().__init__()
        self.options = options
        node_dim = 19 + int(options.ego)
        edge_dim = 17 + int(options.ports)
        self.conv1 = TransformerConv(
            node_dim, 64, heads=2, edge_dim=edge_dim, dropout=0.0
        )
        self.conv2 = TransformerConv(128, 64, heads=2, edge_dim=edge_dim, dropout=0.0)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(256 + edge_dim, 128), torch.nn.ReLU()
        )
        if options.reverse_mp:
            self.reverse1 = TransformerConv(
                node_dim, 64, heads=2, edge_dim=edge_dim, dropout=0.0, root_weight=False
            )
            self.reverse2 = TransformerConv(
                128, 64, heads=2, edge_dim=edge_dim, dropout=0.0, root_weight=False
            )
            self.merge1 = torch.nn.Linear(256, 128, bias=False)
            self.merge2 = torch.nn.Linear(256, 128, bias=False)

    def encode(
        self, x, edge_index, edge_attr, target_index, target_attr, incoming, outgoing
    ):
        direct, direct_attr = edge_index[:, incoming], edge_attr[incoming]
        reverse, reverse_attr = edge_index[:, outgoing].flip(0), edge_attr[outgoing]
        hidden = self.conv1(x, direct, direct_attr)
        if self.options.reverse_mp:
            hidden = self.merge1(
                torch.cat([hidden, self.reverse1(x, reverse, reverse_attr)], dim=1)
            )
        hidden = hidden.relu()
        encoded = self.conv2(hidden, direct, direct_attr)
        if self.options.reverse_mp:
            encoded = self.merge2(
                torch.cat(
                    [encoded, self.reverse2(hidden, reverse, reverse_attr)], dim=1
                )
            )
        encoded = encoded.relu()
        return self.head(
            torch.cat(
                [encoded[target_index[0]], encoded[target_index[1]], target_attr], dim=1
            )
        )


In [ ]:
class BinaryGNN(GraphEncoder):
    """V10: one independent binary risk head, including unmatched laundering."""

    def __init__(self, masses, options, pos_weight=300.0):
        super().__init__(options)
        self.risk_head = torch.nn.Linear(128, 1)
        self.register_buffer(
            "risk_offset", torch.tensor(-np.log(pos_weight), dtype=torch.float32)
        )

    def forward(self, *batch):
        return self.risk_head(self.encode(*batch)).squeeze(-1)

    def scores(self, *batch):
        return (self(*batch) + self.risk_offset).sigmoid()


In [ ]:
"""Load trusted packaged state dictionaries and return a backend-friendly prediction."""

import json
import torch


class Predictor:
    """One immutable model instance with validated, explicit decision thresholds."""

    def __init__(self, family, device="cpu"):
        if family != "binary":
            raise ValueError("This notebook only supports binary")
        self.family = family
        self.device = device
        self.path = ROOT / "models" / f"{family}.pt"
        self.meta = json.loads((ROOT / "models" / f"{family}.json").read_text())
        if sha256(self.path) != self.meta["bundle_sha256"]:
            raise ValueError("Model checksum mismatch")
        self.checkpoint = torch.load(self.path, map_location="cpu", weights_only=True)
        self.options = Options(**self.checkpoint["options"])
        if not self.options.ego or self.options.ports:
            raise ValueError("Bundle supports the selected ego/no-ports checkpoints")
        self.model = (
            BinaryGNN(self.checkpoint["train_masses"], self.options).to(device).eval()
        )
        self.model.load_state_dict(self.checkpoint["state_dict"])
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.threshold = self.checkpoint["validation_threshold"]
        torch.set_num_threads(2)

    def predict(self, graph, threshold=None):
        """Return score and decision for one transaction in its declared snapshot."""
        threshold = self.threshold if threshold is None else float(threshold)
        if not 0 < threshold < 1:
            raise ValueError("threshold must be between 0 and 1")
        with torch.no_grad():
            out = self.model.scores(*graph.tensors(self.device))
        risk = float(out[0][0]) if self.family == "nine_class" else float(out[0])
        result = dict(
            model=self.family,
            model_name="이진분류모델" if self.family == "binary" else "9클래스 모델",
            model_version=self.meta["bundle_sha256"][:16],
            edge_id=graph.edge_id,
            context_end=graph.context_end,
            risk_score=risk,
            threshold=threshold,
            alert=risk >= threshold,
        )
        if self.family == "nine_class":
            probabilities = out[1][0].cpu().numpy()
            result.update(
                pattern_class=API_CLASSES[int(probabilities.argmax())],
                pattern_scores={
                    n: float(p) for n, p in zip(API_CLASSES, probabilities)
                },
                pattern_normal_includes_unmatched=True,
            )
        return result


In [ ]:
if DEVICE.startswith("cuda") and not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is not available; check the installed PyTorch wheel and driver")
predictor = Predictor(FAMILY, device=DEVICE)
print(
    {
        "model": FAMILY,
        "threshold": predictor.threshold,
        "checkpoint": predictor.meta["bundle_sha256"],
    }
)


## 5. 기준 그래프에서 원본 예측 재현

두 NPZ는 원본에서 전처리된 기준 그래프입니다. 합성 예제의 정확도가 아니라 모델 정리 전후 수치 일치를 검증합니다. 허용 오차는 절대 2e-5입니다.

In [ ]:
parity_results = []
for name in ["reference_known", "reference_unmatched"]:
    metadata = json.loads((ROOT / "examples" / f"{name}.json").read_text())
    graph = reference_graph(ROOT / "examples" / f"{name}.npz", metadata)
    result = predictor.predict(graph)
    expected = metadata["expected"][FAMILY]
    np.testing.assert_allclose(
        result["risk_score"], expected["risk_score"], rtol=0, atol=2e-5
    )
    if FAMILY == "nine_class":
        np.testing.assert_allclose(
            list(result["pattern_scores"].values()),
            expected["pattern_scores"],
            rtol=0,
            atol=2e-5,
        )
    parity_results.append(
        {
            "fixture": name,
            "passed": True,
            "risk_score": result["risk_score"],
            "expected": expected["risk_score"],
        }
    )
write_json(ROOT / "outputs" / "reference_validation.json", parity_results)
print(parity_results)


## 6. 입력 → 전처리 → 그래프 → 추론 → JSON

기본 입력은 합성 거래 81건(중복 제거 후 80건)입니다. 아래 셀은 대상 거래를 전부 순회합니다. 운영 데이터로 바꾸면 새 output_dir을 지정하세요. 모델은 한 번 로드해 재사용하고, 그래프 연결은 with 블록 종료 시 닫힙니다. 결과는 JSON Lines로 저장해 대량 응답을 메모리에 모으지 않습니다.

In [ ]:
def run_inference(settings, predictor):
    """Stream one JSON prediction per target; propagate any validation errors."""
    prepared = ensure_snapshot(settings)
    output = Path(settings["output_dir"]) / "predictions.jsonl"
    temporary = output.with_suffix(".jsonl.tmp")
    count = 0
    with GraphSnapshot(prepared) as snapshot, temporary.open(
        "w", encoding="utf-8"
    ) as stream:
        # A separate cursor prevents graph lookups from resetting the target iterator.
        cursor = snapshot.c.cursor()
        try:
            cursor.execute("SELECT edge_id FROM targets ORDER BY edge_id")
            while rows := cursor.fetchmany(256):
                for (edge_id,) in rows:
                    prediction = predictor.predict(snapshot.build(int(edge_id)))
                    stream.write(
                        json.dumps(prediction, ensure_ascii=False, allow_nan=False)
                        + "\n"
                    )
                    count += 1
        finally:
            cursor.close()
    temporary.replace(output)
    return {"predictions": count, "output": str(output)}


run_result = run_inference(settings, predictor)
print(run_result)
with Path(run_result["output"]).open() as stream:
    print(json.dumps(json.loads(next(stream)), ensure_ascii=False, indent=2))


## 실제 입력 예제의 기대 출력 대조

`tests/expected_raw_predictions.json`은 이전 CPU 검증에서 저장한 18건의 기대 출력입니다. 현재 실행 결과와 ID·판정·점수·패턴 확률을 비교합니다. 상대 오차 0, 절대 오차 2e-5를 사용합니다.

In [ ]:
expected = json.loads((ROOT / "tests/expected_raw_predictions.json").read_text())
actual = [json.loads(line) for line in Path(run_result["output"]).read_text().splitlines()]
assert len(actual) == len(expected)
for got, want in zip(actual, expected):
    for key in ["model", "edge_id", "context_end", "alert"]:
        assert got[key] == want[key], (key, got, want)
    np.testing.assert_allclose(got["risk_score"], want["risk_score"], rtol=0, atol=2e-5)
    if FAMILY == "nine_class":
        assert got["pattern_class"] == want["pattern_class"]
        np.testing.assert_allclose(list(got["pattern_scores"].values()), list(want["pattern_scores"].values()), rtol=0, atol=2e-5)
write_json(ROOT / "outputs/raw_input_validation.json", {"passed": True, "device": DEVICE, "rows": len(actual), "absolute_tolerance": 2e-5})
print("Raw input expected-output validation passed:", len(actual), DEVICE)


## 7. 인프라 연결 기준

`Predictor`는 프로세스 시작 시 한 번 생성하고 `predictor.predict(graph)`를 호출합니다. JSON Lines 결과를 API/DB/작업 큐에 연결할 수 있습니다. v10 alert와 v8 pattern_class는 별도 판정으로 유지하세요. 임계값은 테스트에 맞춰 변경하지 않았습니다. 입력 검증 실패나 체크섬 오류는 호출 실패로 처리합니다.

기본 처리 예제는 정확성 검증용 순차 처리입니다. 서버 동시성·대량 배치 처리량은 이번 전달 검증 범위가 아닙니다. 현재 누적 스냅샷 정의를 실시간 거래 직전 문맥으로 바꾸면 모델 성능 재평가가 필요합니다.